# Demo 12e — LSTM and GRU

**Class 12 · G1–G3** (slides: `slides/12_neural_networks.md`)

* **G1:** one LSTM step and one GRU step by hand (numbers of the slides), checked against Keras with the same weights; the gradient along the cell state.
* **G2:** the memory task: accuracy of the RNN, the LSTM and the GRU as the lag grows (3 seeds), and the forget gate of a trained LSTM.
* **G3:** forecasting the temperature of a room one 6 hours ahead: baselines against the three recurrent layers.

In [ ]:
import os

os.environ.setdefault("KERAS_BACKEND", "jax")
import keras
import numpy as np
import polars as pl
from sklearn.linear_model import Ridge

from faa import demo, neural, plotlib
from faa.demo import show, slide

demo.setup("12")
BLUE, ORANGE, GREEN, RED, SKY, PURPLE, GREY = (
    plotlib.BLUE,
    plotlib.ORANGE,
    plotlib.GREEN,
    plotlib.RED,
    plotlib.SKY,
    plotlib.PURPLE,
    plotlib.GREY,
)
SEED = 0

## Slide: One LSTM step by hand (G1)

$d = 1$, $u = 1$ (one unit, so every gate is a number), previous state $h_{t-1} = 0.5$, $c_{t-1} = 1.0$, input $x_t = 1$.
Weights: input gate $(w, v, b) = (1, 1, 0)$, forget gate $(0, 0, 2)$, candidate $(1, 1, 0)$, output gate $(1, 0, 0)$.

In [ ]:
# Keras layout: columns of W, U, b are the gates i, f, g, o
W = np.array([[1.0, 0.0, 1.0, 1.0]])
U = np.array([[1.0, 0.0, 1.0, 0.0]])
b = np.array([0.0, 2.0, 0.0, 0.0])
x, h0, c0 = np.array([1.0]), np.array([0.5]), np.array([1.0])
z = x @ W + h0 @ U + b
i, f, g, o = neural.sigmoid(z[0:1]), neural.sigmoid(z[1:2]), np.tanh(z[2:3]), neural.sigmoid(z[3:4])
c1 = f * c0 + i * g
h1 = o * np.tanh(c1)
slide("One LSTM step by hand")
print(f"pre-activations z = {np.round(z, 3)}")
print(
    f"i = sigma({z[0]:.2f}) = {i[0]:.3f}   f = sigma({z[1]:.2f}) = {f[0]:.3f}   g = tanh({z[2]:.2f}) = {g[0]:.3f}   o = sigma({z[3]:.2f}) = {o[0]:.3f}"
)
print(f"c_t = f c_(t-1) + i g = {f[0]:.3f} x 1.0 + {i[0]:.3f} x {g[0]:.3f} = {c1[0]:.3f}")
print(f"h_t = o tanh(c_t) = {o[0]:.3f} x tanh({c1[0]:.3f}) = {h1[0]:.3f}")
lstm = keras.layers.LSTM(1, return_sequences=True)
lstm.build((None, 1, 1))
lstm.set_weights([W, U, b])
hk = np.array(lstm(x[None, None, :], initial_state=[h0[None], c0[None]])[0, 0])
print(f"Keras LSTM, same weights and state: h_t = {hk[0]:.3f}; difference {abs(hk[0] - h1[0]):.1e}")
vals = {"$i_t$": i[0], "$f_t$": f[0], "$g_t$": g[0], "$o_t$": o[0], "$c_t$": c1[0], "$h_t$": h1[0]}
fig_step, ax = plotlib.fig("col", 1.5)
ax.bar(list(vals), list(vals.values()), color=[GREEN, BLUE, PURPLE, ORANGE, GREY, RED])
for k, v in enumerate(vals.values()):
    ax.text(k, v + 0.03, f"{v:.2f}", ha="center", fontsize=6)
ax.set_ylim(0, 1.9)
ax.set_ylabel("value")
plotlib.save(fig_step, "g1_lstm_step")
hn, cn = neural.lstm_step(x, h0, c0, W, U, b)
print(f"faa.neural.lstm_step: h_t = {hn[0]:.3f}, c_t = {cn[0]:.3f}")

## Slide: One GRU step by hand (G1)

$d = u = 1$, $h_{t-1} = 0.5$, $x_t = 1$. Update gate $z$ (weights 1, 1, bias 0), reset gate $r$ (1, 1, 0), candidate (1, 1, 0).

In [ ]:
Wg = np.array([[1.0, 1.0, 1.0]])
Ug = np.array([[1.0, 1.0, 1.0]])
bg = np.zeros((2, 3))
xg, hg = np.array([1.0]), np.array([0.5])
zx, zh = xg @ Wg + bg[0], hg @ Ug + bg[1]
zg = neural.sigmoid(zx[0] + zh[0])
rg = neural.sigmoid(zx[1] + zh[1])
cand = np.tanh(zx[2] + rg * zh[2])
h_new = zg * hg + (1 - zg) * cand
slide("One GRU step by hand")
print(f"z = sigma(1 + 0.5) = {zg:.3f}   r = sigma(1 + 0.5) = {rg:.3f}")
print(f"candidate = tanh(x W + r (h U)) = tanh(1 + {rg:.3f} x 0.5) = {cand:.3f}")
print(f"h_t = z h_(t-1) + (1 - z) candidate = {zg:.3f} x 0.5 + {1 - zg:.3f} x {cand:.3f} = {h_new[0]:.3f}")
gru = keras.layers.GRU(1, return_sequences=True)
gru.build((None, 1, 1))
gru.set_weights([Wg, Ug, bg])
hk = float(np.array(gru(xg[None, None, :], initial_state=hg[None])[0, 0, 0]))
print(f"Keras GRU, same weights and state: h_t = {hk:.3f}; difference {abs(hk - h_new[0]):.1e}")
print(f"faa.neural.gru_step: {neural.gru_step(xg, hg, Wg, Ug, bg)[0]:.3f}")
f, ax = plotlib.fig("col", 1.5)
vals = {"$z_t$": zg, "$r_t$": rg, "$\\tilde{h}_t$": cand, "$h_t$": h_new[0]}
ax.bar(list(vals), list(vals.values()), color=[GREEN, ORANGE, PURPLE, RED])
for k, v in enumerate(vals.values()):
    ax.text(k, v + 0.03, f"{v:.2f}", ha="center", fontsize=6)
ax.set_ylim(0, 1.1)
ax.set_ylabel("value")
plotlib.save(f, "g1_gru_step")

## Slide: The cell state keeps the gradient (G1)

$\partial c_T/\partial c_t = \prod_{k=t+1}^{T} f_k$: the gradient along the cell state is a product of forget gates, not of weight matrices times derivatives of $\tanh$.
A constant forget gate $f$ over a lag of 20 and of 50 steps:

In [ ]:
rows = [{"forget gate f": fv, "lag 20": fv**20, "lag 50": fv**50} for fv in (0.5, 0.9, 0.95, 0.99, 1.0)]
slide("Forget gate and gradient")
pl.Config.set_fmt_float("mixed")
show(pl.DataFrame(rows), digits=6)
print(
    "Keras initialises the forget-gate bias to 1 (unit_forget_bias): sigma(1) =", round(float(neural.sigmoid(1.0)), 3)
)
f, ax = plotlib.fig("col", 1.6)
lag_axis = np.arange(0, 51)
for fv, c in zip((0.5, 0.9, 0.95, 0.99), (RED, ORANGE, GREEN, BLUE)):
    ax.plot(lag_axis, fv**lag_axis, color=c, label=f"$f$ = {fv}")
ax.set_xlabel("lag")
ax.set_ylabel("$\\prod f_k$")
ax.legend(frameon=False, fontsize=6)
plotlib.save(f, "g1_forget")

## Slide: Memory against the lag (G2)

The first step carries a symbol (one of four), every later step is Gaussian noise (std 1), the label is the symbol: chance is 0.25.
3 000 sequences to train, 1 000 to test, 20 epochs, 32 units, gradient clipped at norm 1, three seeds.

In [ ]:
lags = (10, 20, 40, 60)
acc = {k: {lag: [] for lag in lags} for k in ("rnn", "lstm", "gru")}


def run_lag(lag):
    xs, ys = neural.memory_task(3000, lag, seed=1)
    xt, yt = neural.memory_task(1000, lag, seed=2)
    for kind in acc:
        for s in range(3):
            m = neural.recurrent_model(kind, xs.shape[1:], clipnorm=1.0, seed=s)
            m.fit(xs, ys, epochs=20, batch_size=64, verbose=0)
            acc[kind][lag].append(m.evaluate(xt, yt, verbose=0)[1])


for lag in (10, 20):
    run_lag(lag)

In [ ]:
for lag in (40, 60):
    run_lag(lag)

In [ ]:
rows = [{"lag": lag, **{k.upper(): float(np.mean(acc[k][lag])) for k in acc}} for lag in lags]
slide("Memory against the lag")
show(pl.DataFrame(rows))
f, ax = plotlib.fig("col", 1.8)
for k, c in zip(acc, (RED, BLUE, GREEN)):
    m_ = np.array([np.mean(acc[k][lag]) for lag in lags])
    s_ = np.array([np.std(acc[k][lag]) for lag in lags])
    ax.errorbar(lags, m_, s_, color=c, marker="o", capsize=2, label=k.upper())
ax.axhline(0.25, color=GREY, ls=":", lw=0.8)
ax.set_xlabel("lag (steps)")
ax.set_ylabel("test accuracy")
ax.legend(frameon=False)
plotlib.save(f, "g2_memory_lag")

## Slide: What the forget gate does (G2)

An LSTM trained on the memory task with lag 20; the mean forget gate and the mean input gate over its 32 units at every step of 200 test sequences.

In [ ]:
lag = 20
xs, ys = neural.memory_task(3000, lag, seed=1)
xt, yt = neural.memory_task(200, lag, seed=2)
m = neural.recurrent_model("lstm", xs.shape[1:], clipnorm=1.0, seed=0)
m.fit(xs, ys, epochs=25, batch_size=64, verbose=0)
Wl, Ul, bl = m.layers[0].get_weights()
u = 32
fg, ig, fall = [], [], []
h = np.zeros((len(xt), u))
c = np.zeros((len(xt), u))
for t in range(lag + 1):
    zt = xt[:, t] @ Wl + h @ Ul + bl
    ig.append(neural.sigmoid(zt[:, :u]).mean())
    fgate = neural.sigmoid(zt[:, u : 2 * u])
    fg.append(fgate.mean())
    fall.append(fgate.mean(0))
    h, c = neural.lstm_step(xt[:, t], h, c, Wl, Ul, bl)
fall = np.array(fall)  # steps x units
best = int(np.argmax(np.prod(fall[1:], axis=0)))  # the unit that keeps the most over the lag
slide("Gates of a trained LSTM")
print(f"test accuracy at lag {lag}: {m.evaluate(xt, yt, verbose=0)[1]:.3f}")
print(
    f"best unit: product of its forget gates over steps 1-{lag} = {np.prod(fall[1:, best]):.3f}; mean over all units {np.mean(np.prod(fall[1:], axis=0)):.2e}"
)
print(
    f"mean forget gate: step 0 {fg[0]:.3f}, steps 1-{lag} {np.mean(fg[1:]):.3f} (min {np.min(fg[1:]):.3f}); mean input gate: step 0 {ig[0]:.3f}, steps 1-{lag} {np.mean(ig[1:]):.3f}"
)
f, ax = plotlib.fig("col", 1.6)
ax.plot(fg, color=BLUE, label="forget gate $f_t$")
ax.plot(fall[:, best], color=GREEN, label="forget gate, best unit")
ax.plot(ig, color=ORANGE, label="input gate $i_t$")
ax.set_xlabel("step $t$")
ax.set_ylabel("mean gate value")
ax.legend(frameon=False)
plotlib.save(f, "g2_gates")

## Slide: Forecasting a room temperature (G3)

$x$: the last 24 hours of five room temperatures (standardised on the first 70 % of the hours); target: the temperature of room 1 six hours after the window, as a change from its last value.
Chronological split 70 / 15 / 15 %. Metric: mean absolute error in °C on the last 15 %.
Baselines: persistence (no change), the value 24 hours earlier, ridge regression on the 120 numbers of the window.
Recurrent models: 16 units, early stopping on the validation part, three seeds.

In [ ]:
HOR = 6
Xtr, ytr, Xva, yva, Xte, yte, mu, sd = neural.climate_windows(24, HOR, 5)
dtr, dva, dte = ytr - Xtr[:, -1, 0], yva - Xva[:, -1, 0], yte - Xte[:, -1, 0]
scale = dtr.std()


def mae(p):
    return float(np.abs(np.ravel(p) - dte).mean() * sd)


rows = [
    {
        "model": "persistence",
        "MAE (C)": mae(0 * dte),
        "std over seeds": 0.0,
        "train MAE (C)": float(np.abs(dtr).mean() * sd),
    },
    {
        "model": "same hour yesterday",
        "MAE (C)": mae(Xte[:, -1 + HOR - 24, 0] - Xte[:, -1, 0]),
        "std over seeds": 0.0,
        "train MAE (C)": float(np.abs(Xtr[:, -1 + HOR - 24, 0] - Xtr[:, -1, 0] - dtr).mean() * sd),
    },
    {
        "model": "ridge on the window",
        "MAE (C)": mae(Ridge(1.0).fit(Xtr.reshape(len(Xtr), -1), dtr).predict(Xte.reshape(len(Xte), -1))),
        "std over seeds": 0.0,
        "train MAE (C)": float(
            np.abs(Ridge(1.0).fit(Xtr.reshape(len(Xtr), -1), dtr).predict(Xtr.reshape(len(Xtr), -1)) - dtr).mean() * sd
        ),
    },
]
for kind in ("rnn", "lstm", "gru"):
    maes, train_maes = [], []
    for s in range(3):
        m = neural.recurrent_regressor(kind, Xtr.shape[1:], units=16, seed=s)
        m.fit(
            Xtr,
            dtr / scale,
            epochs=60,
            batch_size=64,
            validation_data=(Xva, dva / scale),
            verbose=0,
            callbacks=[keras.callbacks.EarlyStopping(patience=6, restore_best_weights=True)],
        )
        maes.append(mae(m.predict(Xte, verbose=0).ravel() * scale))
        train_maes.append(float(np.abs(m.predict(Xtr, verbose=0).ravel() * scale - dtr).mean() * sd))
    rows.append(
        {
            "model": kind.upper() + " 16 units",
            "MAE (C)": float(np.mean(maes)),
            "std over seeds": float(np.std(maes)),
            "train MAE (C)": float(np.mean(train_maes)),
        }
    )
slide("Forecasting six hours ahead")
print(f"{len(Xtr)} train, {len(Xva)} validation, {len(Xte)} test windows; standard deviation of room 1: {sd:.2f} C")
print(
    f"room 1 mean (standardised units, 0 = training mean): train {np.mean(Xtr[:, -1, 0]):.2f}, validation {np.mean(Xva[:, -1, 0]):.2f}, test {np.mean(Xte[:, -1, 0]):.2f}"
)
show(pl.DataFrame(rows))
f, ax = plotlib.fig("col", 1.7)
ax.barh(
    [r["model"] for r in rows][::-1],
    [r["MAE (C)"] for r in rows][::-1],
    xerr=[r["std over seeds"] for r in rows][::-1],
    color=[GREY] * 3 + [BLUE] * 3,
)
ax.set_xlabel("test MAE (°C)")
plotlib.save(f, "g3_forecast")